# Lab 1 — Prompt Engineering avec Vertex AI & Gemini

**Cours — Environnements, Infrastructures & Architectures de Déploiement**  
MSc IAG — AIvancity | Certificat 3

### **First Colab**: 
<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/gemini/prompts/intro_prompt_design.ipynb#scrollTo=154137022fb6" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>
<>

## Objectifs
- Maîtriser les patterns de prompting fondamentaux (Persona, Zero-shot, Few-shot, CoT)
- Utiliser le SDK `google-genai` avec Vertex AI (authentification GCP)
- Comparer Gemini 2.5 Flash vs Pro en termes de qualité et de coût
- Implémenter function calling avec Gemini
- Mesurer et optimiser la consommation de tokens

## Prérequis
- Vertex AI Workbench ou Google Colab avec accès GCP
- `PROJECT_ID` GCP configuré
- APIs activées : `Vertex AI API`, `generativelanguage.googleapis.com`

**Durée estimée :** 1h


## 0. Setup — Installation & Authentification

Sur **Vertex AI Workbench**, l'authentification est automatique (compte de service de la VM).  
Sur Colab, exécutez `gcloud auth application-default login` dans un terminal.

In [ ]:
# Installation des dépendances
!pip install -q google-genai python-dotenv


In [ ]:
import os
from google import genai
from google.genai import types
from IPython.display import Markdown, display

# ── Remplacez par votre projet GCP ──────────────────────────────────────
PROJECT_ID = os.environ.get('GOOGLE_CLOUD_PROJECT', 'your-project-id')
LOCATION   = os.environ.get('GOOGLE_CLOUD_REGION', 'us-central1')
# ────────────────────────────────────────────────────────────────────────

client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)
print('✅ Vertex AI client initialisé')
print(f'   Projet : {PROJECT_ID} | Région : {LOCATION}')


In [ ]:
# Modèles disponibles
MODEL_FLASH = 'gemini-2.5-flash'   # Rapide, économique — prod haute volumétrie
MODEL_PRO   = 'gemini-2.5-pro'     # Raisonnement avancé — tâches complexes

def call(prompt, model=MODEL_FLASH, temperature=0.7, system=None):
    """Appel simplifié à Gemini via Vertex AI."""
    cfg = types.GenerateContentConfig(temperature=temperature)
    if system:
        cfg = types.GenerateContentConfig(temperature=temperature,
                                          system_instruction=system)
    resp = client.models.generate_content(model=model, contents=prompt, config=cfg)
    return resp

def pp(resp):
    """Affiche proprement la réponse."""
    display(Markdown(resp.text))

def token_count(prompt, model=MODEL_FLASH):
    """Compte les tokens d'un prompt."""
    r = client.models.count_tokens(model=model, contents=prompt)
    return r.total_tokens

print('✅ Fonctions utilitaires chargées')


---
## Partie 1 — Patterns de Prompting

### 1.1 Persona Pattern
Attribuer un rôle précis au modèle améliore drastiquement la qualité et la tonalité des réponses.

**Syntax :** `Act as [persona]. Perform [task].`

In [ ]:
# ── Exemple fourni ──────────────────────────────────────────────────────
resp = call(
    prompt='Quelles sont les 3 principales différences entre Kubernetes et Docker Swarm ?',
    system='Tu es un architecte cloud senior spécialisé GCP avec 10 ans expérience. '
           'Réponds de façon concise, avec des exemples concrets.'
)
pp(resp)


In [ ]:
# ── TODO 1.1 — Votre persona ─────────────────────────────────────────────
# Choisissez un persona différent (ex: expert sécurité, data scientist, juriste IA)
# et posez-lui une question adaptée à son domaine.
# Observez comment la réponse diffère sans persona.

# VOTRE CODE ICI
persona = "..."
question = "..."

resp_sans_persona = call(question)
resp_avec_persona = call(question, system=persona)

print('=== SANS PERSONA ===')
print(resp_sans_persona.text[:500])
print('\n=== AVEC PERSONA ===')
print(resp_avec_persona.text[:500])


### 1.2 Few-Shot Prompting
Fournir des exemples (input → output) guide le modèle vers le format et le ton attendus.

In [ ]:
# ── Exemple fourni ──────────────────────────────────────────────────────
few_shot_prompt = """Classifie le sentiment des commentaires clients. Réponds uniquement: Positif / Négatif / Neutre

Commentaire: "Le produit est arrivé à l'heure et fonctionne parfaitement."
Sentiment: Positif

Commentaire: "La livraison a pris 3 semaines, c'est inacceptable."
Sentiment: Négatif

Commentaire: "J'ai reçu ma commande."
Sentiment: Neutre

Commentaire: "Je suis vraiment déçu par la qualité, mais le service client était sympa."
Sentiment:"""

resp = call(few_shot_prompt, temperature=0.1)
print('Classification:', resp.text.strip())


In [ ]:
# ── TODO 1.2 ─────────────────────────────────────────────────────────────
# Créez votre propre tâche de few-shot prompting :
# Exemples : extraction d'entités, reformulation, scoring, traduction de format...
# Testez avec 1 exemple vs 3 exemples — mesurez la différence de qualité.

# VOTRE CODE ICI


### 1.3 Chain-of-Thought (CoT)
Demander au modèle de raisonner étape par étape avant de répondre améliore la précision sur les tâches complexes.

In [ ]:
# ── Zero-shot CoT ────────────────────────────────────────────────────────
problem = (
    'Un data center consomme 150 kW. Il tourne 24h/24, 365j/an. '
    'Le coût de l\'électricité est 0.12€/kWh. '
    'Le PUE (Power Usage Effectiveness) est 1.4. '
    'Quel est le coût annuel total en électricité (infra + refroidissement) ?'
)

resp_direct = call(problem + ' Donnez juste le chiffre final.', temperature=0.1)
resp_cot    = call(problem + ' Raisonnez étape par étape, puis donnez la réponse finale.', temperature=0.1)

print('DIRECT  :', resp_direct.text.strip())
print('\nCHAIN-OF-THOUGHT :')
print(resp_cot.text)


In [ ]:
# ── TODO 1.3 ─────────────────────────────────────────────────────────────
# Créez un problème de raisonnement multi-étapes (calcul de coût LLM, logique, SQL...)
# Comparez les réponses avec et sans CoT.
# Essayez aussi le prompt 'Let\'s think step by step' en anglais.

# VOTRE CODE ICI


### 1.4 Prompt Chaining
Décomposer une tâche complexe en étapes séquentielles — la sortie d'une étape devient l'entrée de la suivante.

In [ ]:
# Pipeline : Extraction → Enrichissement → Synthèse
doc = """
Le modèle Gemini 2.5 Pro supporte un contexte de 1 million de tokens, ce qui le rend
particulièrement adapté aux longues conversations, à l'analyse de grandes bases de code
et au traitement de documents volumineux. Ses capacités multimodales lui permettent
de traiter texte, images, audio et vidéo dans un même prompt.
"""

# Étape 1 : Extraction des faits clés
step1 = call(f'Extrait les 3 faits techniques clés de ce texte (bullet points) :\n{doc}',
             temperature=0.3).text
print('ÉTAPE 1 — Faits clés :')
print(step1)

# Étape 2 : Questions que soulèvent ces faits
step2 = call(f'Pour chaque fait, formule une question de compréhension pertinente :\n{step1}',
             temperature=0.5).text
print('\nÉTAPE 2 — Questions :')
print(step2)

# Étape 3 : Synthèse en 1 paragraphe pédagogique
step3 = call(f'Synthétise en 1 paragraphe pédagogique (< 80 mots) :\n{step1}\n\n{step2}',
             temperature=0.4).text
print('\nÉTAPE 3 — Synthèse :')
print(step3)


In [ ]:
# ── TODO 1.4 ─────────────────────────────────────────────────────────────
# Ajoutez une 4e étape : critique-and-revise
# Le modèle évalue la synthèse et la réécrit si elle contient des imprécisions.

# VOTRE CODE ICI


---
## Partie 2 — Contrôle de format & JSON Schema

En production, les sorties doivent être structurées et machine-parseable.

In [ ]:
from pydantic import BaseModel
from typing import List

class DeploymentDecision(BaseModel):
    strategy: str          # 'cloud', 'on_premise', 'hybrid'
    model_recommended: str
    estimated_cost_usd_month: float
    rationale: str
    risks: List[str]

scenario = (
    'Une startup fintech française traite 500 000 requêtes/jour. '
    'Les données contiennent des IBAN et des revenus clients (données sensibles RGPD). '
    'Budget infra : 8 000 €/mois. Latence requise < 300 ms.'
)

resp = client.models.generate_content(
    model=MODEL_FLASH,
    contents=f'Recommande une stratégie de déploiement LLM pour ce scénario : {scenario}',
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        response_schema=DeploymentDecision,
        temperature=0.2,
    )
)
import json
decision = json.loads(resp.text)
print(json.dumps(decision, indent=2, ensure_ascii=False))


In [ ]:
# ── TODO 2 ───────────────────────────────────────────────────────────────
# Créez votre propre schema Pydantic pour un cas d'usage différent :
# Ex : extraction d'entités d'un ticket support, scoring de lead, analyse de contrat...
# Le schema doit avoir au moins 4 champs de types différents (str, int, float, List).

# VOTRE CODE ICI


---
## Partie 3 — Function Calling (ReAct Tool Use)

Gemini peut décider d'appeler des fonctions Python pour obtenir des informations externes.
C'est la base des agents LLM.

In [ ]:
# Outils disponibles pour l'agent
def get_model_pricing(model_name: str) -> dict:
    """Retourne le pricing d'un modèle LLM (USD par 1M tokens)."""
    prices = {
        'gemini-2.5-flash': {'input': 0.075, 'output': 0.30, 'context_k': 1000},
        'gemini-2.5-pro':   {'input': 1.25,  'output': 5.00, 'context_k': 1000},
        'gpt-4o':           {'input': 2.50,  'output': 10.0, 'context_k': 128},
        'claude-sonnet-4':  {'input': 3.00,  'output': 15.0, 'context_k': 200},
    }
    return prices.get(model_name.lower(), {'error': 'Modèle inconnu'})

def calculate_monthly_cost(model_name: str, daily_requests: int,
                           avg_input_tokens: int, avg_output_tokens: int) -> dict:
    """Calcule le coût mensuel estimé pour un volume donné."""
    pricing = get_model_pricing(model_name)
    if 'error' in pricing:
        return pricing
    monthly_req = daily_requests * 30
    cost_input  = (monthly_req * avg_input_tokens / 1_000_000) * pricing['input']
    cost_output = (monthly_req * avg_output_tokens / 1_000_000) * pricing['output']
    return {
        'model': model_name,
        'monthly_requests': monthly_req,
        'cost_input_usd': round(cost_input, 2),
        'cost_output_usd': round(cost_output, 2),
        'total_usd': round(cost_input + cost_output, 2)
    }

# Test rapide des outils
print(get_model_pricing('gemini-2.5-flash'))
print(calculate_monthly_cost('gemini-2.5-flash', 10000, 500, 200))


In [ ]:
# Laisser Gemini décider quels outils appeler
tool_config = types.ToolConfig(
    function_calling_config=types.FunctionCallingConfig(mode='AUTO')
)

query = (
    'Notre application fait 15 000 requêtes par jour avec en moyenne '
    '800 tokens en entrée et 300 tokens en sortie. '
    'Compare le coût mensuel entre Gemini 2.5 Flash et GPT-4o. '
    'Recommande le meilleur choix.'
)

resp = client.models.generate_content(
    model=MODEL_FLASH,
    contents=query,
    config=types.GenerateContentConfig(
        tools=[get_model_pricing, calculate_monthly_cost],
        tool_config=tool_config,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(maximum_remote_calls=5),
        temperature=0.2
    )
)
pp(resp)
